# Lab 1 at home · Two tables of your own in DuckDB

**Due:** Sunday, October 25 at 11:59 PM · **Submit:** your repo URL on CourseWorks

In class you loaded one table (penguins), found a problem and fixed it. Now do the same with **two related tables you choose**. In Session 2 we'll join them, so pick two that can be connected.

You will:
1. Pick two related public datasets and upload them
2. Load both into DuckDB and check what DuckDB decided about each column
3. Find the column the two tables share (the **key**) and check how well they match
4. Write five SQL queries that help explain your data
5. Write observations and a dataset card for each table

## 1. Pick two related tables

**Related** means they share a column you could match rows on: an ID, a name, a ZIP code, a country code, a date.

**Example pairs**
- **World Bank:** any indicator download (e.g. GDP per country per year) comes with a second file of country details (region, income group). The shared column is the country code.
- **Our World in Data:** two charts for the same countries, e.g. life expectancy and health spending. Shared columns: country and year.
- **NYC Open Data:** 311 complaints and population by ZIP code, or the street tree census and neighborhood boundaries. Shared column: ZIP code or neighborhood.
- **Your own idea:** any two public tables that share a column.

**Rules**
- **Public, open data only.** Your repo is public, and later this data goes to a free AI tier. Nothing from work, nothing personal, nothing behind a login.
- **CSV format.** If a file is Excel, open it and save it as CSV first.
- **Each file under 100 MB.** Between 1,000 and 1,000,000 rows is a good size for the main table; the second can be small.
- **Reuse allowed.** Check each license or terms page, and note it in your dataset card.

**Places to look:** [NYC Open Data](https://opendata.cityofnewyork.us) · [data.gov](https://data.gov) · [World Bank Open Data](https://data.worldbank.org) · [Our World in Data](https://ourworldindata.org) · [Kaggle Datasets](https://www.kaggle.com/datasets) (check each license)

## 2. Upload them

1. Download both CSVs to your computer.
2. In this Codespace, drag both files into the **`data`** folder in the file list on the left.
3. Rename each to something short with no spaces, e.g. `gdp.csv` and `countries.csv` (right-click → **Rename**).
4. Fill in the cell below and run it. Table names should be short, lowercase, no spaces: you'll type them in your SQL.

In [51]:
FILE_A  = "restaurants.csv"     # <- your main table's file name
TABLE_A = "restaurants"                   # <- a short name for it, e.g. "gdp"

FILE_B  = "zillow_rent.csv"   # <- the related table's file name
TABLE_B = "rent"                 # <- a short name for it, e.g. "countries"

In [52]:
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_A = ROOT / "data" / FILE_A
DATA_B = ROOT / "data" / FILE_B

missing = [p.name for p in (DATA_A, DATA_B) if not p.exists()]
if missing:
    print("Can't find:", ", ".join(missing))
    print("Files in data/ right now:")
    for f in sorted((ROOT / "data").iterdir()):
        print("  ", f.name)
else:
    con = duckdb.connect(str(ROOT / "lab1_home.duckdb"))
    for p in (DATA_A, DATA_B):
        print(f"Found data/{p.name} ({p.stat().st_size / 1_000_000:.1f} MB)")
    print("Connected.")

Found data/restaurants.csv (10.0 MB)
Found data/zillow_rent.csv (10.1 MB)
Connected.


## 3. First look

Preview the first 10 rows of each raw file before loading.

In [53]:
con.sql(f"SELECT * FROM read_csv('{DATA_A}') LIMIT 10")

┌───────┬──────────┬────────────────────────────────────────────────┬────────┬─────────┬───────────────────────────────────────────────────────────┬─────────────┬───────────────────────────────────────────────────┬──────────┬────────────┬─────────────┐
│  id   │ position │                      name                      │ score  │ ratings │                         category                          │ price_range │                   full_address                    │ zip_code │    lat     │     lng     │
│ int64 │  int64   │                    varchar                     │ double │  int64  │                          varchar                          │   varchar   │                      varchar                      │ varchar  │   double   │   double    │
├───────┼──────────┼────────────────────────────────────────────────┼────────┼─────────┼───────────────────────────────────────────────────────────┼─────────────┼───────────────────────────────────────────────────┼──────────┼────────────┼───

In [54]:
con.sql(f"SELECT * FROM read_csv('{DATA_B}') LIMIT 10")

┌──────────┬──────────┬────────────┬────────────┬───────────┬─────────┬──────────────┬────────────────────────────────────────────────┬──────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬──────

## 4. Load both and check the types

Load each file into a table, then `DESCRIBE` it. Compare each column's type with what it *should* be. A number or date column stored as `VARCHAR` (text) is the same kind of problem you fixed in class.

### TABLE_A
ZIP code clean

In [55]:
for table, path in ((TABLE_A, DATA_A), (TABLE_B, DATA_B)):
    con.sql(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM read_csv('{path}')")
    rows = con.sql(f"SELECT count(*) FROM {table}").fetchone()[0]
    print(f"{table}: {rows:,} rows loaded")

restaurants: 63,469 rows loaded
rent: 8,340 rows loaded


In [56]:
con.sql(f"DESCRIBE {TABLE_A}")

┌──────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name  │ column_type │  null   │   key   │ default │  extra  │
│   varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ position     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ name         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ score        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ ratings      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ category     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ price_range  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ full_address │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ zip_code     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ lat          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ lng 

In [57]:
con.sql(f"DESCRIBE {TABLE_B}")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ RegionID    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ SizeRank    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ RegionName  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ RegionType  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ StateName   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ State       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ City        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Metro       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ CountyName  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ 2015-01-31  │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│     ·       │   · 

PP notes: I was wondering why zip code is varchar instead of bigint. It turns out some area's zip code would look like 08071, if I change in into bigint, it would become 80701.

`SUMMARIZE` profiles every column at once: min, max, how many distinct values, and `null_percentage` (how much is missing).

In [58]:
con.sql(f"SUMMARIZE {TABLE_A}")

┌──────────────┬─────────────┬─────────────────────────────────────────────────┬──────────────────────────────────────────────┬───────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬───────┬─────────────────┐
│ column_name  │ column_type │                       min                       │                     max                      │ approx_unique │        avg         │        std         │        q25         │        q50         │        q75         │ count │ null_percentage │
│   varchar    │   varchar   │                     varchar                     │                   varchar                    │     int64     │      varchar       │      varchar       │      varchar       │      varchar       │      varchar       │ int64 │  decimal(9,2)   │
├──────────────┼─────────────┼─────────────────────────────────────────────────┼──────────────────────────────────────────────┼───────────────┼────────────────────┼───────────

PP notes: ZIP codes and zero coordinates in the restaurant dataset looks weird.

In [59]:
con.sql(f"SUMMARIZE {TABLE_B}")

┌─────────────┬─────────────┬───────────────────┬────────────────────┬───────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬───────┬─────────────────┐
│ column_name │ column_type │        min        │        max         │ approx_unique │        avg         │        std         │        q25         │        q50         │        q75         │ count │ null_percentage │
│   varchar   │   varchar   │      varchar      │      varchar       │     int64     │      varchar       │      varchar       │      varchar       │      varchar       │      varchar       │ int64 │  decimal(9,2)   │
├─────────────┼─────────────┼───────────────────┼────────────────────┼───────────────┼────────────────────┼────────────────────┼────────────────────┼────────────────────┼────────────────────┼───────┼─────────────────┤
│ RegionID    │ BIGINT      │ 58197             │ 870188             │          9684 │ 84745.66630695443  │ 43853.49654181889  │

In [60]:
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT RegionName) AS unique_zip,
        COUNT(*) FILTER (
            WHERE RegionName IS NULL
        ) AS missing_zip,
        COUNT(*) FILTER (
            WHERE RegionName IS NOT NULL
              AND NOT regexp_full_match(RegionName, '[0-9]{5}')
        ) AS invalid_zip,
        COUNT(*) - COUNT(DISTINCT RegionName) AS duplicate_zip
    FROM rent
""")

┌────────────┬────────────┬─────────────┬─────────────┬───────────────┐
│ total_rows │ unique_zip │ missing_zip │ invalid_zip │ duplicate_zip │
│   int64    │   int64    │    int64    │    int64    │     int64     │
├────────────┼────────────┼─────────────┼─────────────┼───────────────┤
│       8340 │       8340 │           0 │           0 │             0 │
└────────────┴────────────┴─────────────┴─────────────┴───────────────┘

## 5. Fix what loaded wrong (if anything)

If every type looks right, write "No fixes needed" in the text cell after the next one and skip ahead.

Otherwise, open the CSV (double-click it in the `data` folder) and find **how missing values are written** in the columns that came in as text. Edit the `nullstr` list below to match, and run it for the table that needs fixing.

### TABLE_A
ZIP code clean

In [61]:
# Change TABLE_A / DATA_A to TABLE_B / DATA_B if it's the second table that needs fixing
# con.sql(f"""
    #CREATE OR REPLACE TABLE {TABLE_A} AS
    #SELECT * FROM read_csv('{DATA_A}', nullstr = ['NA', 'N/A', ''])
#""")
# con.sql(f"DESCRIBE {TABLE_A}")
pass

PP notes: didn't use the cell above, for there is nothing wrong in describ.

In [62]:
con.sql("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE zip_code IS NULL
        ) AS missing_zip,

        COUNT(*) FILTER (
            WHERE regexp_full_match(zip_code, '[0-9]{4}')
        ) AS four_digit_zip,

        COUNT(*) FILTER (
            WHERE regexp_full_match(zip_code, '[0-9]{5}')
        ) AS valid_format_zip,

        COUNT(*) FILTER (
            WHERE zip_code IS NOT NULL
              AND NOT regexp_full_match(zip_code, '[0-9]{4}')
              AND NOT regexp_full_match(zip_code, '[0-9]{5}')
        ) AS other_invalid_zip

    FROM restaurants
""")

┌────────────┬─────────────┬────────────────┬──────────────────┬───────────────────┐
│ total_rows │ missing_zip │ four_digit_zip │ valid_format_zip │ other_invalid_zip │
│   int64    │    int64    │     int64      │      int64       │       int64       │
├────────────┼─────────────┼────────────────┼──────────────────┼───────────────────┤
│      63469 │         516 │            220 │            61808 │               925 │
└────────────┴─────────────┴────────────────┴──────────────────┴───────────────────┘

In [63]:
con.sql("""
    SELECT
        zip_code,
        COUNT(*) AS count
    FROM restaurants
    WHERE zip_code IS NOT NULL
      AND NOT regexp_full_match(zip_code, '[0-9]{4}')
      AND NOT regexp_full_match(zip_code, '[0-9]{5}')
    GROUP BY zip_code
    ORDER BY count DESC
    LIMIT 30
""")

┌────────────┬───────┐
│  zip_code  │ count │
│  varchar   │ int64 │
├────────────┼───────┤
│ US         │     4 │
│ 75237-2426 │     3 │
│ 75070-2908 │     3 │
│ 75042-6829 │     3 │
│ 75013-4637 │     3 │
│ 20010-2513 │     3 │
│ 55811-5605 │     2 │
│ 22042-3002 │     2 │
│ 60050-1422 │     2 │
│ 22315-3223 │     2 │
│     ·      │     · │
│     ·      │     · │
│     ·      │     · │
│ 98008-3908 │     2 │
│ 54303-4708 │     2 │
│ 54913-9294 │     2 │
│ 23321-2006 │     2 │
│ 54304-4616 │     2 │
│ 22203-1812 │     2 │
│ 20112-8685 │     2 │
│ 25414-4964 │     2 │
│ 84102-2804 │     2 │
│ 22025-1997 │     2 │
└────────────┴───────┘
  30 rows  2 columns
  (20 shown)         

In [64]:
con.sql("""
    SELECT
        CASE
            WHEN zip_code IS NULL THEN 'Missing'
            WHEN regexp_full_match(zip_code, '[0-9]{5}')
                THEN 'Standard ZIP'
            WHEN regexp_full_match(zip_code, '[0-9]{5}-[0-9]{4}')
                THEN 'ZIP+4'
            WHEN regexp_full_match(zip_code, '[0-9]{4}')
                THEN 'Four-digit ZIP'
            ELSE 'Other'
        END AS zip_type,
        COUNT(*) AS count
    FROM restaurants
    GROUP BY zip_type
    ORDER BY count DESC
""")

┌────────────────┬───────┐
│    zip_type    │ count │
│    varchar     │ int64 │
├────────────────┼───────┤
│ Standard ZIP   │ 61808 │
│ ZIP+4          │   776 │
│ Missing        │   516 │
│ Four-digit ZIP │   220 │
│ Other          │   149 │
└────────────────┴───────┘

In [65]:
con.sql("""
    SELECT
        zip_code,
        COUNT(*) AS count
    FROM restaurants
    WHERE zip_code IS NOT NULL
      AND NOT regexp_full_match(zip_code, '[0-9]{5}')
      AND NOT regexp_full_match(zip_code, '[0-9]{5}-[0-9]{4}')
      AND NOT regexp_full_match(zip_code, '[0-9]{4}')
    GROUP BY zip_code
    ORDER BY count DESC
""")

┌───────────┬───────┐
│ zip_code  │ count │
│  varchar  │ int64 │
├───────────┼───────┤
│ US        │     4 │
│ 961       │     2 │
│ 949       │     2 │
│ 0         │     2 │
│ Milwaukee │     1 │
│ 260623720 │     1 │
│ 985979425 │     1 │
│ 989026200 │     1 │
│ 989023065 │     1 │
│ 981773121 │     1 │
│     ·     │     · │
│     ·     │     · │
│     ·     │     · │
│ 233204871 │     1 │
│ 233225233 │     1 │
│ 233143210 │     1 │
│ 84115     │     1 │
│ 840782701 │     1 │
│ 953       │     1 │
│ 78749.0   │     1 │
│ 77070.0   │     1 │
│ TX 75028  │     1 │
│ None      │     1 │
└───────────┴───────┘
      143 rows     
     (20 shown)     

In [66]:
con.sql("""
    WITH classified AS (
        SELECT
            zip_code,
            CASE
                WHEN regexp_full_match(zip_code, '[0-9]{9}')
                    THEN 'Nine-digit ZIP'
                WHEN regexp_full_match(zip_code, '[0-9]{3}')
                    THEN 'Three-digit ZIP'
                WHEN regexp_full_match(zip_code, '[0-9]{5}[.]0')
                    THEN 'Decimal format'
                WHEN regexp_full_match(zip_code, '[A-Z]{2} [0-9]{5}')
                    THEN 'State + ZIP'
                WHEN regexp_full_match(zip_code, '[A-Za-z]+')
                    THEN 'Text'
                ELSE 'Other unrecognized'
            END AS issue_type
        FROM restaurants
        WHERE zip_code IS NOT NULL
          AND NOT regexp_full_match(zip_code, '[0-9]{5}')
          AND NOT regexp_full_match(zip_code, '[0-9]{5}-[0-9]{4}')
          AND NOT regexp_full_match(zip_code, '[0-9]{4}')
    )

    SELECT
        issue_type,
        COUNT(*) AS count
    FROM classified
    GROUP BY issue_type
    ORDER BY count DESC
""")

┌────────────────────┬───────┐
│     issue_type     │ count │
│      varchar       │ int64 │
├────────────────────┼───────┤
│ Nine-digit ZIP     │   123 │
│ Text               │     8 │
│ Three-digit ZIP    │     7 │
│ Other unrecognized │     7 │
│ Decimal format     │     3 │
│ State + ZIP        │     1 │
└────────────────────┴───────┘

In [67]:
con.sql("""
    SELECT
        name,
        zip_code,
        full_address,
        CASE
            WHEN regexp_full_match(zip_code, '[0-9]{3}')
                THEN 'Three-digit ZIP'
            WHEN regexp_full_match(zip_code, '[0-9]{5}[.]0')
                THEN 'Decimal format'
            WHEN regexp_full_match(zip_code, '[A-Z]{2} [0-9]{5}')
                THEN 'State + ZIP'
            WHEN regexp_full_match(zip_code, '[A-Za-z]+')
                THEN 'Text'
            ELSE 'Other unrecognized'
        END AS issue_type
    FROM restaurants
    WHERE zip_code IS NOT NULL
      AND NOT regexp_full_match(zip_code, '[0-9]{5}')
      AND NOT regexp_full_match(zip_code, '[0-9]{5}-[0-9]{4}')
      AND NOT regexp_full_match(zip_code, '[0-9]{4}')
      AND NOT regexp_full_match(zip_code, '[0-9]{9}')
    ORDER BY issue_type, zip_code
""").df()

,name,zip_code,full_address,issue_type
0,Mad Over Curry,77070.0,"10758 Cypress Creek Parkway, Houston, TX, 77070.0",Decimal format
1,Gorditas Doña Lula (E Expy 83),78596.0,"1919 W Expy 83, Weslaco, TX, 78596.0",Decimal format
2,Starbucks (SR 71 &amp; US 290),78749.0,"7015 West Hwy 290, Austin, TX, 78749.0",Decimal format
3,Health Prep Meals,0,"205 South Cage Boulevard, Pharr, TX, 0",Other unrecognized
4,Alpha Dessert Juice Cafe,0,"11246 Wilcrest Drive, Houston, TX, 0",Other unrecognized
5,Nacho Libre (Guaynabo),000961,"Los Trailers, Puerto Rico, , 000961",Other unrecognized
6,Amai Crepe (Rockville),208500,"199 J East Montgomery Ave, Rockville, MD, 208500",Other unrecognized
7,Earnest Donuts - Lewisville,75056,"4740 State Hwy 121, Lewisville, TX, 75056",Other unrecognized
8,WU KONG BBQ,84115,"3390 South State Street, Ste 22 ., Salt Lake C...",Other unrecognized
9,County Clare Irish Inn and Pub,United States,"1234 N Astor St, Milwaukee, WI, United States",Other unrecognized


I kept valid five-digit ZIP codes and standardized nine-digit ZIP codes (with or without a hyphen) by extracting the first five digits. I excluded records with missing ZIP codes or other invalid formats, including four-digit codes, because I could not confirm whether they had lost a leading zero or were simply incorrect.

This cleaning process reduced the dataset from 63,469 to 62,707 records, removing 762 records in total.

In [68]:
con.sql("""
    CREATE OR REPLACE TABLE restaurants_clean AS
    SELECT
        *,
        CASE
            WHEN regexp_full_match(zip_code, '[0-9]{5}')
                THEN zip_code
            WHEN regexp_full_match(zip_code, '[0-9]{5}-[0-9]{4}')
                THEN LEFT(zip_code, 5)
            WHEN regexp_full_match(zip_code, '[0-9]{9}')
                THEN LEFT(zip_code, 5)
        END AS clean_zip
    FROM restaurants
    WHERE regexp_full_match(zip_code, '[0-9]{5}')
       OR regexp_full_match(zip_code, '[0-9]{5}-[0-9]{4}')
       OR regexp_full_match(zip_code, '[0-9]{9}')
""")

In [69]:
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT clean_zip) AS unique_zip_codes,
        COUNT(*) FILTER (
            WHERE clean_zip IS NULL
               OR NOT regexp_full_match(clean_zip, '[0-9]{5}')
        ) AS invalid_zip
    FROM restaurants_clean
""")

┌────────────┬──────────────────┬─────────────┐
│ total_rows │ unique_zip_codes │ invalid_zip │
│   int64    │      int64       │    int64    │
├────────────┼──────────────────┼─────────────┤
│      62707 │             2252 │           0 │
└────────────┴──────────────────┴─────────────┘

### TABLE_A
lon & lat clean

### TABLE_A
LON & LAT clean

In [70]:
con.sql("""
    CREATE OR REPLACE TABLE restaurants_final AS
    SELECT *
    FROM restaurants_clean
    WHERE lat != 0 AND lng != 0
""")

In [71]:
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE lat = 0 OR lng = 0
        ) AS invalid_coordinates
    FROM restaurants_final
""")

┌────────────┬─────────────────────┐
│ total_rows │ invalid_coordinates │
│   int64    │        int64        │
├────────────┼─────────────────────┤
│      62667 │                   0 │
└────────────┴─────────────────────┘

### CREATE NEW TABLE

In [72]:
con.sql(f"""
    COPY restaurants_final
    TO '{ROOT / "data" / "restaurants_clean.csv"}'
    (HEADER, DELIMITER ',')
""")

**What did you fix, and how did you know?** 

**TABLE_A**

Since Table A was collected through web scraping, I found inconsistencies in the **ZIP code formats**. Some records used five-digit ZIP codes, while others used nine-digit formats. I standardized them into five-digit ZIP codes for consistency.
I also removed records with missing or invalid values, such as zero coordinates, and excluded unreliable ZIP codes, including entries containing state or country names and incomplete numerical codes. This helped improve data consistency and reliability for further analysis.

I also found that some records had **latitude** or **longitude** values of zero, which likely indicated missing location data. I removed these records to ensure more reliable geographic analysis.

**TABLE_B**

The ZIP codes in Table B are consistently formatted, with no missing or duplicate values. However, **rental data coverage** varies across months, with some periods having more missing values than others.

Since the dataset provides monthly rental prices over time, it could be used for a temporal analysis of rent trends in neighborhoods where the restaurants are located, assuming those restaurants remained in operation during the study period.


## 6. Find the key: how do the two tables connect?

The **key** is the column both tables share. Type its name in each table below. It may be spelled differently in each, e.g. `Country Code` in one and `iso3` in the other.

The next cell counts the distinct key values in each table and how many of them appear in both. (It treats both as text, so a ZIP code stored as a number still matches one stored as text.)

In [73]:
TABLE_A = "restaurants_final"

KEY_A = "clean_zip"   # <- the shared column's name in your main table
KEY_B = "RegionName"   # <- its name in the second table

In [74]:
con.sql(f"""
    WITH a AS (SELECT DISTINCT CAST("{KEY_A}" AS VARCHAR) AS k FROM {TABLE_A}),
         b AS (SELECT DISTINCT CAST("{KEY_B}" AS VARCHAR) AS k FROM {TABLE_B})
    SELECT
        (SELECT count(*) FROM a)                           AS keys_in_a,
        (SELECT count(*) FROM b)                           AS keys_in_b,
        (SELECT count(*) FROM a WHERE k IN (SELECT k FROM b)) AS keys_in_both
""")

┌───────────┬───────────┬──────────────┐
│ keys_in_a │ keys_in_b │ keys_in_both │
│   int64   │   int64   │    int64     │
├───────────┼───────────┼──────────────┤
│      2251 │      8340 │         1584 │
└───────────┴───────────┴──────────────┘

In [75]:
con.sql("""
    SELECT DISTINCT clean_zip
    FROM restaurants_final
    WHERE clean_zip NOT IN (
        SELECT RegionName FROM rent
    )
    ORDER BY clean_zip
    LIMIT 10
""")

┌───────────┐
│ clean_zip │
│  varchar  │
├───────────┤
│ 00000     │
│ 00646     │
│ 00917     │
│ 00918     │
│ 00920     │
│ 00921     │
│ 00923     │
│ 00924     │
│ 00925     │
│ 00926     │
└───────────┘
   10 rows 

PP notes: During the key matching process, I discovered an issue missed in the initial cleaning: ZIP code `00000` had passed the five-digit format check but is not a valid US ZIP code. This shows that a correctly formatted value is not necessarily valid, and additional validation is needed beyond checking data types and formats.

**Describe the relationship** (double-click to edit):

- **Key column:** clean_zip in Table A (Uber Eats Restaurants) and RegionName in Table B (Zillow Rent).
- **How they relate:** The relationship is many-to-one. Multiple restaurants can share the same ZIP code, while Zillow has only one row per ZIP code, containing monthly rental prices.
- **How well they match:** Table A has 2,251 unique ZIP codes, and Table B has 8,340. Of these, 1,584 appear in both tables (70.4% of Table A's ZIP codes). There are 667 ZIP codes only in Table A and 6,756 only in Table B. This may be due to differences in geographic coverage and data availability between the two datasets.

For example, ZIP code `00000` appeared in Table A but not in Table B. This revealed an issue missed during the initial cleaning: although `00000` follows the five-digit format, it is not a valid US ZIP code. Other unmatched ZIP codes, such as those in Puerto Rico, may result from differences in geographic coverage between the two datasets.



In [76]:
con.sql("""
    CREATE OR REPLACE TABLE restaurants_verified AS
    SELECT *
    FROM restaurants_final
    WHERE clean_zip != '00000'
""")

In [47]:
con.sql(f"""
    COPY restaurants_verified
    TO '{ROOT / "data" / "restaurants_verified.csv"}'
    (HEADER, DELIMITER ',')
""")

### rerun cell

In [77]:
TABLE_A = "restaurants_verified"
KEY_A = "clean_zip"  # <- the shared column's name in your main table
KEY_B = "RegionName"   # <- its name in the second table

In [78]:
con.sql(f"""
    WITH a AS (SELECT DISTINCT CAST("{KEY_A}" AS VARCHAR) AS k FROM {TABLE_A}),
         b AS (SELECT DISTINCT CAST("{KEY_B}" AS VARCHAR) AS k FROM {TABLE_B})
    SELECT
        (SELECT count(*) FROM a)                           AS keys_in_a,
        (SELECT count(*) FROM b)                           AS keys_in_b,
        (SELECT count(*) FROM a WHERE k IN (SELECT k FROM b)) AS keys_in_both
""")

┌───────────┬───────────┬──────────────┐
│ keys_in_a │ keys_in_b │ keys_in_both │
│   int64   │   int64   │    int64     │
├───────────┼───────────┼──────────────┤
│      2250 │      8340 │         1584 │
└───────────┴───────────┴──────────────┘

## 7. Five queries that explain your data

Write one query of each kind. Under each, double-click the text cell and write one line: **what does this result tell us?** Use either table.

Tip: if a column name has spaces or capitals, wrap it in double quotes: `"Country Name"`. Start from a query in the in-class notebook and change one thing.

### Query 1 · Size and shape
*How many rows, and over what time period or area?* Useful SQL: `count(*), min(), max()`

In [79]:
con.sql("""
    SELECT
        COUNT(*) AS total_restaurants,
        COUNT(DISTINCT clean_zip) AS unique_zip_codes,
        MIN(lat) AS min_latitude,
        MAX(lat) AS max_latitude,
        MIN(lng) AS min_longitude,
        MAX(lng) AS max_longitude
    FROM restaurants_verified
""")

┌───────────────────┬──────────────────┬──────────────┬──────────────┬───────────────┬───────────────┐
│ total_restaurants │ unique_zip_codes │ min_latitude │ max_latitude │ min_longitude │ max_longitude │
│       int64       │      int64       │    double    │    double    │    double     │    double     │
├───────────────────┼──────────────────┼──────────────┼──────────────┼───────────────┼───────────────┤
│             62666 │             2250 │     18.33924 │     48.96395 │    -123.84124 │     -65.96904 │
└───────────────────┴──────────────────┴──────────────┴──────────────┴───────────────┴───────────────┘

**What this tells us:** The cleaned dataset contains 62,666 restaurant records across 2,250 unique ZIP codes.

### Query 2 · Filter
*Which rows meet a condition you care about?* Useful SQL: `WHERE`

In [85]:
con.sql("""
    SELECT
        name,
        full_address,
        clean_zip,
        lat,
        lng
    FROM restaurants_verified
    WHERE clean_zip LIKE '100%'
       OR clean_zip LIKE '101%'
       OR clean_zip LIKE '102%'
       OR clean_zip LIKE '103%'
       OR clean_zip LIKE '104%'
       OR clean_zip LIKE '111%'
       OR clean_zip LIKE '112%'
       OR clean_zip LIKE '113%'
       OR clean_zip LIKE '114%'
       OR clean_zip LIKE '116%'
       OR clean_zip IN ('11004', '11005')
""").df()

,name,full_address,clean_zip,lat,lng
0,Starbucks (1920 East Riverside Dr.),"1920 East Riverside Dr., Austin, TX, 11342",11342,30.242500,-97.728090
1,Lucky Moo Foods - Wattala,"28/13 Matagorda Lane, Matagorda,hendala, Watta...",11300,29.753499,-95.365805


In [84]:
con.sql("""
    SELECT
        name,
        category,
        price_range,
        clean_zip,
        score,
        full_address
    FROM restaurants_verified
    WHERE lat BETWEEN 30.10 AND 30.55
      AND lng BETWEEN -97.95 AND -97.55
""")

┌───────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────┬─────────────┬───────────┬────────┬──────────────────────────────────────────────────────┐
│                             name                              │                                           category                                           │ price_range │ clean_zip │ score  │                     full_address                     │
│                            varchar                            │                                           varchar                                            │   varchar   │  varchar  │ double │                       varchar                        │
├───────────────────────────────────────────────────────────────┼──────────────────────────────────────────────────────────────────────────────────────────────┼─────────────┼───────────┼────────┼────────────────────────────────────────────────────

**What this tells us:** 

I tested the geographic coverage of this nationwide restaurant dataset by filtering for New York City and Austin. The NYC ZIP code filter returned only two records, both with inconsistent location information, while the Austin coordinate filter returned 3,010 records. This suggests uneven geographic coverage, with Austin much better represented than NYC in this dataset.

### Query 3 · Group
*How does a count or average differ across categories?* Useful SQL: `GROUP BY`

In [86]:
con.sql("""
    SELECT
        clean_zip,
        COUNT(*) AS restaurant_count
    FROM restaurants_verified
    WHERE lat BETWEEN 30.10 AND 30.55
      AND lng BETWEEN -97.95 AND -97.55
    GROUP BY clean_zip
    ORDER BY clean_zip
""")

┌───────────┬──────────────────┐
│ clean_zip │ restaurant_count │
│  varchar  │      int64       │
├───────────┼──────────────────┤
│ 11342     │                1 │
│ 75038     │               10 │
│ 78610     │                2 │
│ 78613     │              187 │
│ 78617     │                2 │
│ 78634     │               27 │
│ 78641     │                7 │
│ 78645     │                2 │
│ 78652     │                4 │
│ 78653     │                4 │
│   ·       │                · │
│   ·       │                · │
│   ·       │                · │
│ 78749     │               39 │
│ 78750     │               73 │
│ 78751     │              171 │
│ 78752     │               72 │
│ 78753     │              123 │
│ 78754     │               15 │
│ 78756     │               43 │
│ 78757     │               89 │
│ 78758     │              115 │
│ 78759     │              141 │
└───────────┴──────────────────┘
  57 rows (20 shown) 2 columns

**What this tells us:** 

The number of restaurants recorded varies across ZIP codes in the Austin area. However, since this is a web-scraped dataset with uneven geographic coverage, these counts reflect the collected sample rather than the actual distribution of restaurants.

### Query 4 · Rank
*What are the top or bottom 10 of something?* Useful SQL: `ORDER BY ... LIMIT 10`

In [90]:
con.sql("""
    WITH locations AS (
        SELECT
            TRIM(SPLIT_PART(full_address, ',', -3)) AS city,
            TRIM(SPLIT_PART(full_address, ',', -2)) AS state
        FROM restaurants_verified
        WHERE full_address IS NOT NULL
    )
    SELECT
        city,
        state,
        COUNT(*) AS restaurant_count
    FROM locations
    WHERE city != ''
      AND state != ''
    GROUP BY city, state
    ORDER BY restaurant_count DESC
    LIMIT 10
""")

┌─────────────┬─────────┬──────────────────┐
│    city     │  state  │ restaurant_count │
│   varchar   │ varchar │      int64       │
├─────────────┼─────────┼──────────────────┤
│ Houston     │ TX      │             3972 │
│ San Antonio │ TX      │             2401 │
│ Austin      │ TX      │             2356 │
│ Seattle     │ WA      │             1733 │
│ Dallas      │ TX      │             1717 │
│ Washington  │ DC      │             1454 │
│ Fort Worth  │ TX      │             1144 │
│ Portland    │ OR      │              926 │
│ Richmond    │ VA      │              806 │
│ Plano       │ TX      │              785 │
└─────────────┴─────────┴──────────────────┘
  10 rows                        3 columns

**What this tells us:** 

The dataset shows uneven geographic coverage, with six of the ten most represented cities located in Texas. New York City is notably absent from the ranking despite its large restaurant population. This suggests potential sampling bias in the web-scraped data, meaning the recorded counts should not be treated as representative of actual restaurant distributions.

### Query 5 · Quality check
*What's missing, duplicated or impossible?* Useful SQL: `count(*) - count(column), count(DISTINCT ...)`

In [91]:
con.sql("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(DISTINCT id) AS unique_restaurants,
        COUNT(*) - COUNT(DISTINCT id) AS duplicate_ids
    FROM restaurants_verified
""")

┌───────────────┬────────────────────┬───────────────┐
│ total_records │ unique_restaurants │ duplicate_ids │
│     int64     │       int64        │     int64     │
├───────────────┼────────────────────┼───────────────┤
│         62666 │              62666 │             0 │
└───────────────┴────────────────────┴───────────────┘

In [94]:
con.sql("""
    SELECT
        name,
        full_address,
        COUNT(*) AS occurrences
    FROM restaurants_verified
    GROUP BY name, full_address
    HAVING COUNT(*) > 1
    ORDER BY occurrences DESC
""").df()

,name,full_address,occurrences
0,SportClub,"750 N Jefferson St, Milwaukee, WI, 53202",2
1,Subway (3227 Old Forest Rd),"3227 Old Forest Rd, Lynchburg, VA, 24501",2
2,Acapulco Mexican Restaurant,"800 Lafayette Ave, Moundsville, WV, 26041",2
3,7-Eleven (6975 Capitol Blvd),"6975 Capitol Blvd, Tumwater, WA, 98501",2
4,Saigon Deli,"4142 Brooklyn Ave NE, Seattle, WA, 98105",2
...,...,...,...
78,America's Best Wings,"2302 S State Hwy 121, Lewisville, TX, 75067",2
79,Addison Sushi 469,"4995, Addison, TX, 75001",2
80,Big A’s,"955 South State Highway 46, New Braunfels, TX,...",2
81,WooMeeOk Korean BBQ,"10560 Walnut St, Dallas, TX, 75243",2


**What this tells us:** 


Although all 62,666 records have unique IDs, I found 83 pairs of records sharing the same restaurant name and address. This suggests potential duplicate entries in the web-scraped dataset, which could lead to overestimating restaurant counts.

## 8. Three observations

Specific things you noticed that you'd want to check before trusting this data. Point to numbers.

1. Inconsistent Data Formatting: Since the restaurant dataset was collected through web scraping, the data contains inconsistent formats and irregular entries. For example, ZIP codes appear in both five-digit and nine-digit formats, while some entries even contain state or country names. In total, 1,661 records initially failed the five-digit ZIP code check, showing the challenges of cleaning web-scraped data. 
2. Uneven Geographic Coverage: Although the dataset is labeled as covering the United States, the geographic distribution of its records is highly uneven. Six of the top ten cities with the most records are in Texas, while New York City has almost no reliable records in our checks. This suggests that the dataset is not geographically representative and may reflect the scraping strategy rather than actual restaurant distribution.
3. Potential Duplicate Records: The dataset contains repeated restaurant entries despite having unique IDs. Among the 62,666 cleaned records, 83 pairs share the same restaurant name and address. This suggests possible duplication during data collection, which could affect the accuracy of restaurant counts.

**One question you'd want a non-technical person to be able to ask of this data:**

How can we assess and improve the reliability of imperfect datasets?



## 9. Dataset cards

| | Main table | Second table |
|---|---|---|
| **Dataset name** |Uber Eats USA Restaurants|Zillow Observed Rent Index (ZORI), ZIP Code Level|
| **Source (link)** |Kaggle|Zillow Research|
| **Who collected it, and why** |Ahmed Shahriar Sakib collected restaurant information from the Uber Eats website using web scraping for public data analysis.|Zillow Research produces rental market estimates to track housing costs and trends across the US.|
| **License or terms of use** |Publicly available on Kaggle; exact dataset license needs verification on the dataset page.|Free for public research use under Zillow's Terms of Use, with attribution required.|
| **Rows × columns** |Original: 63,469 × 11; Cleaned: 62,666 × 12|8,340 × 150|
| **Time period covered** |Historical snapshot; exact collection dates are not specified in the CSV.|January 2015 – September 2026|
| **How often it's updated** |The publisher described a weekly scraping/update schedule; current update frequency is unverified.|Monthly|

## 10. Submit

1. Run the cell below to close the connection.
2. Fill in the lifecycle map of your data in `labs/lab1.md`.
3. **Save this notebook (Cmd+S)**, then commit and push: **Source Control** → message `Lab 1` → **Commit** → **Sync Changes**.
4. Check on GitHub that both CSVs, both notebooks and `labs/lab1.md` are there, with results visible.
5. Paste your repo URL into the Lab 1 assignment on CourseWorks.

In [95]:
con.close()
print("Done. Save, commit, push, and submit your repo URL.")

Done. Save, commit, push, and submit your repo URL.
